# 3.0 - HOG + SVM

HOG features (Dalal & Triggs, 2005 default detector parameters --
see src/features.py) trained with SVM, evaluated on validation and
then once on the test set for the final reported metric.

In [ ]:
# Clone the repo and install dependencies.
# facenet-pytorch needs --no-deps: Colab has no prebuilt wheels for the
# old numpy/Pillow versions it normally asks for. (Not strictly needed
# for this notebook since we don't run MTCNN here, but kept for a
# consistent, working environment across all notebooks.)

!git clone https://github.com/laianemuckler/liveness-detection.git
%cd liveness-detection

!pip install -r requirements.txt --quiet
!pip install facenet-pytorch==2.6.0 --no-deps --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd

from src.config import TRAIN_DIR, VAL_DIR, TEST_DIR, DRIVE_ROOT
from src.dataset import load_processed_split
from src.features import extract_features
from src.modeling.train import train_svm
from src.modeling.predict import evaluate, log_experiment

In [ ]:
# Load the already-processed (MTCNN-aligned) image paths + labels.
# Same processed data used by the LBP notebook -- no need to redo
# preprocessing.

train_paths, y_train = load_processed_split(TRAIN_DIR)
val_paths, y_val = load_processed_split(VAL_DIR)
test_paths, y_test = load_processed_split(TEST_DIR)

print(f"Train: {len(train_paths)} | Validation: {len(val_paths)} | Test: {len(test_paths)}")

In [ ]:
RUNS_DIR = os.path.join(DRIVE_ROOT, 'experiments', 'hog_svm', 'runs')
os.makedirs(RUNS_DIR, exist_ok=True)

## Experiment: HOG default (Dalal & Triggs, 2005)

Only one configuration here: unlike LBP, there is no anti-spoofing
specific reference tuning these parameters (see docs/decisions.md),
so the well-established default detector parameters are used as-is.

In [ ]:
EXP_ID = 'hog_svm_01_default'
exp_dir = os.path.join(RUNS_DIR, EXP_ID)
os.makedirs(exp_dir, exist_ok=True)

train_feat_path = os.path.join(exp_dir, 'X_train.npy')
val_feat_path = os.path.join(exp_dir, 'X_val.npy')
test_feat_path = os.path.join(exp_dir, 'X_test.npy')

# Extract features (or load them if already saved from a previous run)
if os.path.exists(train_feat_path):
    print("Features already extracted, loading from disk...")
    X_train = np.load(train_feat_path)
    X_val = np.load(val_feat_path)
    X_test = np.load(test_feat_path)
else:
    print("Extracting HOG features...")
    X_train = extract_features(train_paths, 'hog_default')
    X_val = extract_features(val_paths, 'hog_default')
    X_test = extract_features(test_paths, 'hog_default')

    np.save(train_feat_path, X_train)
    np.save(val_feat_path, X_val)
    np.save(test_feat_path, X_test)

print(f"Feature shape: {X_train.shape}")

In [ ]:
# Train SVM on the training set
model, scaler = train_svm(X_train, y_train, kernel='rbf', C=1.0)

In [ ]:
# Evaluate on validation, log to experiment_log.csv
val_results = evaluate(model, scaler, X_val, y_val)
print(f"Validation -> HTER: {val_results['HTER']*100:.2f}% | AUC: {val_results['AUC']:.4f}")

log_experiment(
    exp_id=EXP_ID,
    metodo='HOG+SVM',
    feature_config='hog_default',
    modelo_config='kernel=rbf, C=1.0',
    hter_val=val_results['HTER'],
    auc_val=val_results['AUC'],
)

## Final evaluation on the test set (done once)

In [ ]:
test_results = evaluate(model, scaler, X_test, y_test)

print(f"FINAL TEST RESULT ({EXP_ID})")
print(f"HTER: {test_results['HTER']*100:.2f}%")
print(f"AUC:  {test_results['AUC']:.4f}")

log_experiment(
    exp_id=EXP_ID + '_FINAL_TEST',
    metodo='HOG+SVM',
    feature_config='hog_default',
    modelo_config='kernel=rbf, C=1.0',
    hter_test=test_results['HTER'],
    auc_test=test_results['AUC'],
    obs='Final reported test metric for HOG+SVM'
)

## Save predictions for later analysis

In [ ]:
df_predictions = pd.DataFrame({
    'image_path': test_paths,
    'label_true': y_test,
    'label_pred': test_results['y_pred'],
    'score': test_results['y_scores'],
})

predictions_path = os.path.join(exp_dir, 'predictions_test.csv')
df_predictions.to_csv(predictions_path, index=False)
print(f"Predictions saved to {predictions_path}")